![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Redis Agent Memory with the NVIDIA NeMo Agent Toolkit

## Introduction

AI agents need **memory** to feel coherent: they should remember who the user is, what they prefer, and what was said earlier — across turns and across sessions. The [**NVIDIA NeMo Agent Toolkit**](https://github.com/NVIDIA/NeMo-Agent-Toolkit) (NAT) is a framework for building and orchestrating agents, and [**nemo-agent-toolkit-redis**](https://github.com/redis-developer/nemo-agent-toolkit-redis) plugs Redis-backed memory into it.

In this recipe we wire NAT up to the open-source, self-hosted [**Redis Agent Memory Server**](https://github.com/redis/agent-memory-server) — a memory service that automatically extracts, stores, and semantically retrieves facts from conversations.

## What we'll build

A chat agent whose memory is fully managed for us:

- **Working (short-term) memory** — the live conversation, scoped to a session, with a TTL.
- **Long-term memory** — durable facts (preferences, names, etc.) the server extracts from the conversation in the background and recalls later via semantic search.

We use the toolkit's `redis_agent_memory_auto_memory` workflow wrapper, which on every turn hydrates the prompt with relevant memory and captures the new turn automatically — no manual save/load calls in our agent code.

### Architecture

```
  your notebook ──run_workflow()──▶ NAT auto-memory workflow
                                          │
                                          ▼
                              Redis Agent Memory Server  (localhost:8000)
                                          │
                                          ▼
                                    Redis Stack  (localhost:6379)
```

## Let's Begin

> **NOTE:** This notebook drives real services (Redis + the Agent Memory Server via Docker) and calls OpenAI, so it is intended to run locally rather than in Google Colab or a CI pipeline.

## Prerequisites

- **Docker** (to run Redis Stack and the Agent Memory Server locally).
- An **OpenAI API key** — the Agent Memory Server uses it for extraction/embeddings, and NAT uses it for the chat LLM.

In [1]:
# NBVAL_SKIP
# nvidia-nat-langchain registers the OpenAI->LangChain LLM client that the
# chat_completion function uses; without it you get a KeyError at run time.
%pip install -q nemo-agent-toolkit-redis nvidia-nat-langchain requests

Note: you may need to restart the kernel to use updated packages.


## Set environment variables

In [2]:
import os, getpass, warnings

# Silence two upstream deprecation warnings (nvidia-nat-core's LangChain .text()
# call, and the agent-memory client's get_working_memory) so demo output stays clean.
warnings.filterwarnings("ignore", message=r".*\.text\(\) as a method is deprecated.*")
warnings.filterwarnings("ignore", message=r".*get_working_memory is deprecated.*")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# The self-hosted Agent Memory Server we start below
os.environ["REDIS_AGENT_MEMORY_URL"] = "http://localhost:8000"
os.environ["REDIS_AGENT_MEMORY_NAMESPACE"] = "nat-auto-memory"
os.environ["NAT_OPENAI_MODEL"] = "gpt-4o-mini"

# The open-source server runs with auth disabled, so no auth header is needed.
AUTH_HEADERS: dict = {}

## Start the Agent Memory Server (self-hosted)

We start Redis Stack and the open-source Agent Memory Server with Docker. The server runs with `DISABLE_AUTH=true` for local development and enables background long-term-memory extraction with the `discrete` strategy.

> The alternative is the project's `compose.yml`: `docker compose up -d`.

In [3]:
# NBVAL_SKIP
import subprocess
import time, requests

def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}\n{p.stderr.strip()}")
    print(p.stdout.strip() or 'ok')

# Redis Stack (search + JSON) for the memory server to use as its store
sh('docker rm -f nat-redis 2>/dev/null; '
   'docker run -d --name nat-redis -p 6379:6379 redis/redis-stack:7.4.0-v8')

# Agent Memory Server, talking to the Redis container over the host network.
# `-e OPENAI_API_KEY` with no value tells docker to pass it through from our
# environment, so the key never lands on the command line (ps / shell history).
sh('docker rm -f nat-agent-memory 2>/dev/null; '
   'docker run -d --name nat-agent-memory -p 8000:8000 '
   '-e REDIS_URL=redis://host.docker.internal:6379 '
   '-e OPENAI_API_KEY '
   '-e DISABLE_AUTH=true '
   '-e LONG_TERM_MEMORY=true '
   '--add-host=host.docker.internal:host-gateway '
   'redislabs/agent-memory-server:0.14.0 '
   'agent-memory api --host 0.0.0.0 --port 8000 --task-backend=asyncio')

# First run pulls images, so poll until the server accepts connections.
for _ in range(30):
    try:
        r = requests.get('http://localhost:8000/v1/health', timeout=2)
        if r.ok:
            print('Agent Memory Server status:', r.status_code, r.json())
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Agent Memory Server didn't come up. Check: docker logs nat-agent-memory")

2be306c58d000d0bc14a0291fe144b84742b31e1bb451a4b6c6b240c6ec5473f
c7eb418aae0c6505ce801b114f97e7d25e639b7d5e88c6baa97371eb3acb9d3a
Agent Memory Server status: 200 {'now': 1783721388658}


## Define the NAT workflow

NAT is a full-featured agent toolkit and it manages its agents through workflow config yaml files. You can
learn more about config files on Nvidia's docs site [here](https://docs.nvidia.com/nemo/agent-toolkit/1.2/workflows/about/index.html).

The workflow config ties three things together: the chat LLM (`openai_llm`), the memory backend pointed at our server (`redis_agent_memory_backend`), and the `redis_agent_memory_auto_memory` wrapper that does hydration + capture. Environment variables (`${...}`) are resolved from the values we set above.

In [4]:
config_yaml = """general:
  telemetry:
    enabled: false

llms:
  openai_llm:
    _type: openai
    model_name: ${NAT_OPENAI_MODEL:-gpt-4o-mini}

functions:
  assistant_chat:
    _type: chat_completion
    llm_name: openai_llm
    system_prompt: >-
      You are a helpful assistant. When memory context is provided, use it to
      answer consistently about the user's preferences and prior facts.

memory:
  redis_ltm:
    _type: redis_agent_memory_backend
    base_url: ${REDIS_AGENT_MEMORY_URL:-http://localhost:8000}
    default_namespace: ${REDIS_AGENT_MEMORY_NAMESPACE:-nat-auto-memory}

workflow:
  _type: redis_agent_memory_auto_memory
  description: >-
    A chat agent that uses Redis Agent Memory working memory plus memory_prompt
    hydration on every turn.
  inner_agent_name: assistant_chat
  memory_name: redis_ltm
  default_user_id: demo-user
  default_session_id: demo-session
  memory_prompt:
    optimize_query: false
    long_term_search:
      limit: 5
  working_memory:
    namespace: ${REDIS_AGENT_MEMORY_NAMESPACE:-nat-auto-memory}
    model_name: ${NAT_OPENAI_MODEL:-gpt-4o-mini}
    ttl_seconds: 86400
    long_term_memory_strategy:
      strategy: discrete
"""

with open("nat_config.yml", "w") as f:
    f.write(config_yaml)
print("wrote nat_config.yml")

wrote nat_config.yml


## Run the agent

`nat.utils.run_workflow` runs a single turn. We pass `session_kwargs` so NAT knows which `user_id` / `conversation_id` (session) the memory belongs to.

In [5]:
# NBVAL_SKIP
from pathlib import Path

from nat.utils import run_workflow

CONFIG_FILE = Path("nat_config.yml").resolve()


async def chat(prompt: str, user_id: str = "demo-user", conversation_id: str = "demo-session") -> str:
    """Run one turn through the NAT auto-memory workflow.

    NAT maps user_id -> Redis Agent Memory user_id and conversation_id -> session_id,
    so working memory is hydrated and turns are captured automatically.
    """
    result = await run_workflow(
        config_file=CONFIG_FILE,
        prompt=prompt,
        to_type=str,
        session_kwargs={"conversation_id": conversation_id, "user_id": user_id},
    )
    print(f"User: {prompt}")
    print(f"Assistant: {result}\n")
    return result

In [6]:
# NBVAL_SKIP
# A multi-turn conversation. Turn 3 relies on memory captured in turns 1-2.
await chat("Hi! My name is Justin and my favorite city is Lisbon.")
await chat("I'm a vegetarian, by the way.")
# New turn -> the auto-memory wrapper hydrates prior facts from Redis Agent Memory
await chat("Where should I plan a food trip, and what should I keep in mind?")

Package metadata not found for nvidia-nat
Package metadata not found for nvidia-nat
Package metadata not found for nvidia-nat
/Users/justin.cechmanek/.pyenv/versions/3.11.9/envs/redis-ai-res/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/justin.cechmanek/.pyenv/versions/3.11.9/envs/redis-ai-res/lib/python3.11/site-packages/nat/builder/function.py:380: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  return await self._ainvoke_fn(value)


User: Hi! My name is Justin and my favorite city is Lisbon.
Assistant: Hi Justin! Lisbon is a beautiful city with a rich history and vibrant culture. What do you love most about it?



/Users/justin.cechmanek/.pyenv/versions/3.11.9/envs/redis-ai-res/lib/python3.11/site-packages/nat/builder/function.py:380: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  return await self._ainvoke_fn(value)


User: I'm a vegetarian, by the way.
Assistant: That's great to know, Justin! Lisbon has some fantastic vegetarian options. Have you tried any local vegetarian dishes there or have any favorites?

User: Where should I plan a food trip, and what should I keep in mind?
Assistant: Planning a food trip can be an exciting adventure! Here are some tips and suggestions:

### Destinations
1. **Barcelona, Spain**: Known for its tapas, paella, and fresh produce.
2. **Tokyo, Japan**: Explore vegetarian-friendly options like ramen, sushi, and seasonal vegetables.
3. **Bologna, Italy**: Famous for its pasta dishes and rich culinary traditions.
4. **Bangkok, Thailand**: Offers vibrant street food with plenty of vegetarian choices.
5. **Mexico City, Mexico**: Discover traditional dishes like tamales and enchiladas that can be made vegetarian.

### Tips for Planning
1. **Research Local Cuisine**: Look into vegetarian-friendly dishes from the region you'll be visiting.
2. **Use Apps and Websites**: Chec

/Users/justin.cechmanek/.pyenv/versions/3.11.9/envs/redis-ai-res/lib/python3.11/site-packages/nat/builder/function.py:380: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  return await self._ainvoke_fn(value)


"Planning a food trip can be an exciting adventure! Here are some tips and suggestions:\n\n### Destinations\n1. **Barcelona, Spain**: Known for its tapas, paella, and fresh produce.\n2. **Tokyo, Japan**: Explore vegetarian-friendly options like ramen, sushi, and seasonal vegetables.\n3. **Bologna, Italy**: Famous for its pasta dishes and rich culinary traditions.\n4. **Bangkok, Thailand**: Offers vibrant street food with plenty of vegetarian choices.\n5. **Mexico City, Mexico**: Discover traditional dishes like tamales and enchiladas that can be made vegetarian.\n\n### Tips for Planning\n1. **Research Local Cuisine**: Look into vegetarian-friendly dishes from the region you'll be visiting.\n2. **Use Apps and Websites**: Check out platforms like HappyCow to find vegetarian or vegan restaurants.\n3. **Join Food Tours**: Consider joining a food tour that caters to vegetarian diets to discover hidden gems.\n4. **Cultural Considerations**: Be aware of local customs regarding food and dining

Notice the third answer reflects facts (name, favorite city, vegetarian) from earlier turns even though we never passed them back in — the auto-memory wrapper retrieved them from Redis Agent Memory.

## Inspect long-term memory

Facts are promoted to long-term memory in the background. We can query them directly through the server's REST API.

**NOTE:** Memory extraction is an async background task so these memories may not populate right away.
If the request in the cell below isn't returning any memories try waiting a few seconds and running it again.

In [7]:
# NBVAL_SKIP
# Inspect what got promoted to long-term memory via the Agent Memory REST API.
base = os.environ["REDIS_AGENT_MEMORY_URL"].rstrip("/")
namespace = os.environ.get("REDIS_AGENT_MEMORY_NAMESPACE", "nat-auto-memory")

resp = requests.post(
    f"{base}/v1/long-term-memory/search",
    headers={"Content-Type": "application/json", **AUTH_HEADERS},
    json={"text": "favorite city and diet", "namespace": {"eq": namespace}, "limit": 5},
    timeout=30,
)
resp.raise_for_status()
for m in resp.json().get("memories", []):
    print(f"- [{m.get('memory_type')}] {m.get('text')}")

- [semantic] User's favorite city is Lisbon, Portugal.
- [episodic] On July 10, 2026, User asked for suggestions to plan a vegetarian-friendly food trip.
- [semantic] User follows a vegetarian diet.


## Cleanup

In [8]:
# NBVAL_SKIP
# `|| true` keeps this idempotent — re-running when the containers are already
# gone shouldn't raise.
sh('docker rm -f nat-agent-memory nat-redis 2>/dev/null || true')

nat-agent-memory
nat-redis


## Summary

We gave a NeMo Agent Toolkit agent persistent memory with **~30 lines of config and no memory plumbing** in the agent itself. The self-hosted [Redis Agent Memory Server](https://github.com/redis/agent-memory-server) handled extraction, storage, and semantic recall.

Ready for production? The next notebook, `07_nemo_agent_toolkit_redis_cloud.ipynb`, runs the same agent against **managed [Redis Cloud Agent Memory](https://redis.io/agent-memory/)** — no server to operate.